<a href="https://colab.research.google.com/github/lord-of-the-strings/Data-Science/blob/main/airports.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study: Geospatial Risk Mitigation for Global Aerospace Infrastructure
### Datasets: Global Facility Registry & World Tectonic Plate Boundaries

## 1. Problem Statement & Parameters
Aerospace ground installations, telemetry tracking nodes, and launch sites require high structural stability. This analysis maps the spatial proximity of global major infrastructure assets relative to active tectonic plate boundaries to identify systemic seismic risk.

* **Buffer Parameter:** $50\text{ km}$ uniform geographic radius.
* **Target Coordinates:** Active large and medium infrastructure nodes ($N = 5,276$).

## 2. Geoprocessing & Projection Pipeline
1. **Coordinate Alignment:** Both raw datasets were initialized under standard GPS angular degrees (**EPSG:4326**).
2. **Metric Re-projection:** To execute precise uniform distance calculations without spherical angular distortion, data layers were re-projected into the World Mercator metric projection (**EPSG:3395**).
3. **Proximity Modeling:** A $50,000\text{ meter}$ polygon vector buffer was computed around linear plate boundaries.
4. **Spatial Intersection:** A point-in-polygon spatial join (`gpd.sjoin`) isolated nodes falling within danger vectors.

## 3. Results & Civil Engineering Insights
* **Risk Density:** $364$ facilities out of $5,276$ ($6.9\%$) sit within the immediate seismic danger zone.
* **Primary Geographic Vulnerabilities:** The highest concentrations of at-risk nodes are concentrated within **CN, VE, US, TR, and TW**, directly tracking major plate collision lines and the Pacific Ring of Fire.

## 4. Operational Takeaway
Any future launch vehicle test facilities or satellite communications arrays constructed within these zones must incorporate advanced seismic dampening foundations, or be explicitly prioritized for structural telemetry monitoring.

In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
url="https://raw.githubusercontent.com/davidmegginson/ourairports-data/master/airports.csv"
df_raw=pd.read_csv(url)
df_clean = df_raw[df_raw['type'].isin(['large_airport', 'medium_airport'])].copy()
df_clean=df_clean.dropna(subset=['latitude_deg','longitude_deg'])
geometry = [Point(lon, lat) for lon, lat in zip(df_clean['longitude_deg'], df_clean['latitude_deg'])]
spaceports = gpd.GeoDataFrame(df_clean, geometry=geometry, crs="EPSG:4326")
print("Spaceport GeoDataFrame successfully built!")
print(f"Total active facilities mapped: {len(spaceports)}")
print("CRS Verified:", spaceports.crs)

Spaceport GeoDataFrame successfully built!
Total active facilities mapped: 5276
CRS Verified: EPSG:4326


In [4]:
print(spaceports['type'].value_counts())

type
medium_airport    4097
large_airport     1179
Name: count, dtype: int64


In [6]:
import numpy as np
metric=spaceports.to_crs(3395)
metric['distance_equator_km'] = np.abs(metric['geometry'].y) / 1000.0
closest_facilites = metric[['name', 'iso_country', 'distance_equator_km']].sort_values(by='distance_equator_km')
print(closest_facilites.head(5))

                                                    name iso_country  \
28339                                   Mbandaka Airport          CD   
30825                                  Laikipia Air Base          KE   
31202                      Entebbe International Airport          UG   
82069                                   Tebelian Airport          ID   
57205  Macapá - Alberto Alcolumbre International Airport          BR   

       distance_equator_km  
28339             2.498979  
30825             3.209308  
31202             4.686802  
82069             4.999506  
57205             5.602136  


In [8]:
plates_url = "https://raw.githubusercontent.com/fraxen/tectonicplates/master/GeoJSON/PB2002_boundaries.json"
plates=gpd.read_file(plates_url)
print("Tectonic Plates Layer Loaded Successfully!")
print("Plates Geometry Type:", plates.geom_type.unique())
print("Plates Initial CRS:", plates.crs)

Tectonic Plates Layer Loaded Successfully!
Plates Geometry Type: ['LineString']
Plates Initial CRS: EPSG:4326


In [10]:
plates_metric=plates.to_crs(3395)
plates_metric['danger_zone']=plates_metric['geometry'].buffer(50000)
danger=plates_metric.set_geometry('danger_zone')
facilities_at_risk = gpd.sjoin(metric,danger, how='inner', predicate='within')
print("Spatial Join Execution Complete!")
print(f"Total facilities sitting within a 50km seismic risk zone: {len(facilities_at_risk)}")

Spatial Join Execution Complete!
Total facilities sitting within a 50km seismic risk zone: 364


In [11]:
risk_by_country = facilities_at_risk['iso_country'].value_counts()
print("Top 5 Countries with Infrastructure in Seismic Danger Zones:")
print(risk_by_country.head(5))

Top 5 Countries with Infrastructure in Seismic Danger Zones:
iso_country
CN    38
VE    29
US    20
TR    19
TW    18
Name: count, dtype: int64
